# Solutions — Advanced Window Functions (Medium 17)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
franchises   = spark.table("samples.bakehouse.sales_franchises")
trips        = spark.table("samples.nyctaxi.trips")

## Solution 1 — rank() vs dense_rank()

In [ ]:
w = Window.partitionBy("franchiseID").orderBy(F.col("totalPrice").desc())
result_1 = (
    transactions
    .select(
        "franchiseID","product","totalPrice",
        F.rank().over(w).alias("rnk"),
        F.dense_rank().over(w).alias("dense_rnk")
    )
    .orderBy("franchiseID", F.col("totalPrice").desc())
)

## Solution 2 — ntile Bucketing (Quartiles within Franchise)

In [ ]:
w = Window.partitionBy("franchiseID").orderBy("totalPrice")
result_2 = (
    transactions
    .withColumn("quartile", F.ntile(4).over(w))
    .groupBy("franchiseID","quartile")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("franchiseID","quartile")
)

## Solution 3 — percent_rank and cume_dist

In [ ]:
w = Window.partitionBy("franchiseID").orderBy("totalPrice")
result_3 = (
    transactions
    .select(
        "franchiseID","transactionID","totalPrice",
        F.round(F.percent_rank().over(w), 4).alias("pct_rank"),
        F.round(F.cume_dist().over(w), 4).alias("cum_dist")
    )
    .orderBy("franchiseID","totalPrice")
)

## Solution 4 — Lead: Next Price in Partition

In [ ]:
w = Window.partitionBy("franchiseID").orderBy("dateTime")
result_4 = (
    transactions
    .withColumn("next_price", F.lead("totalPrice", 1).over(w))
    .withColumn("price_diff", F.round(F.col("next_price") - F.col("totalPrice"), 2))
    .select("franchiseID","transactionID","totalPrice","next_price","price_diff")
    .orderBy("franchiseID","dateTime")
)

## Solution 5 — 3-Row Moving Average

In [ ]:
w = Window.partitionBy("franchiseID").orderBy("dateTime").rowsBetween(-2, 0)
result_5 = (
    transactions
    .select(
        "franchiseID","transactionID","dateTime","totalPrice",
        F.round(F.avg("totalPrice").over(w), 2).alias("moving_avg_3")
    )
    .orderBy("franchiseID","dateTime")
)

## Solution 6 — Range-Based Running Total (Taxi)

In [ ]:
w = Window.partitionBy("pickup_zip").orderBy(
    F.col("tpep_pickup_datetime").cast("long")
).rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_6 = (
    trips
    .select(
        "pickup_zip","tpep_pickup_datetime","fare_amount",
        F.round(F.sum("fare_amount").over(w), 2).alias("running_total")
    )
    .orderBy("pickup_zip","tpep_pickup_datetime")
)

## Solution 7 — First and Last Price per Franchise

In [ ]:
result_7 = (
    transactions
    .groupBy("franchiseID")
    .agg(
        F.first("totalPrice").alias("first_price"),
        F.last("totalPrice").alias("last_price")
    )
    .orderBy("franchiseID")
)

## Solution 8 — Multiple Window Specs, Filter rn <= 5

In [ ]:
w_rn  = Window.partitionBy("franchiseID").orderBy(F.col("dateTime").desc())
w_sum = Window.partitionBy("franchiseID")
w_avg = Window.partitionBy("franchiseID").orderBy("dateTime").rowsBetween(-2, 0)
result_8 = (
    transactions
    .withColumn("rn",              F.row_number().over(w_rn))
    .withColumn("franchise_total", F.round(F.sum("totalPrice").over(w_sum), 2))
    .withColumn("moving_avg",      F.round(F.avg("totalPrice").over(w_avg), 2))
    .filter(F.col("rn") <= 5)
    .select("franchiseID","transactionID","totalPrice","franchise_total","moving_avg","rn")
    .orderBy("franchiseID","rn")
)